In [22]:
!python --version

Python 3.13.11


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("geo_data.csv")
df.head()

,Unnamed: 0,address,name_ru,avg_rating,rubrics,lat,lon,geocode_status
0,0,"г. Москва, пр-кт Волгоградский, д. 46 Б",Стратегия Оценки,5.0,Оценочная компания;Экспертиза;Строительная экс...,NaN,NaN,not_found
1,1,",Москва, Западный административный округ, райо...",One and Double,1.0,Кофейня,NaN,NaN,not_found
2,2,"Адрес: Московская область, Люберецкий район, Б...",Газмагистраль,5.0,"АГНС, АГЗС, АГНКС",NaN,NaN,not_found
3,3,Москва,Экспериментальный квартал,5.0,Достопримечательность,55.625578,37.606392,success
4,4,Москва,ОкнаДел,5.0,Окна;Изготовление витражей;Реставрационная мас...,55.625578,37.606392,success


In [3]:
df = df.dropna(subset = ["lat"])
df = df[df["address"] != "Москва"]
df.head()

,Unnamed: 0,address,name_ru,avg_rating,rubrics,lat,lon,geocode_status
16,16,"Москва, 1-й Автозаводский проезд, 4к1",Абрикосик,2.0,Массажный салон,55.703938,37.656436,success
17,17,"Москва, 1-й Автозаводский проезд, 5",Чайхана Азия,2.0,Кафе,55.704831,37.657289,success
18,18,"Москва, 1-й Автозаводский проезд, 5",RenarDance,5.0,Школа танцев,55.704831,37.657289,success
20,20,"Москва, 1-й Амбулаторный проезд, 8с1",Колледж автомобильного транспорта № 9,5.0,Колледж,55.811594,37.532844,success
21,21,"Москва, 1-й Балтийский переулок, 3/25",ХуанХэ,5.0,Ресторан;Кафе,55.810501,37.518897,success


In [4]:
df = df.drop("geocode_status", axis = 1)

In [19]:
from geopy.distance import EARTH_RADIUS


def haversin(theta):
    return (1 - np.cos(theta)) / 2.0

def haversine_2D_mat(data1, data2):
    lats1, lons1 = data1['lat'].values, data1['lon'].values
    phis1, lambs1 = np.radians(lats1).reshape(-1, 1), np.radians(lons1).reshape(-1, 1)

    lats2, lons2 = data2['lat'].values, data2['lon'].values
    phis2, lambs2 = np.radians(lats2).reshape(-1, 1), np.radians(lons2).reshape(-1, 1)

    deltas_lats = phis1 - phis2.T
    deltas_lons = lambs1 - lambs2.T

    cos_phis1 = np.cos(phis1)
    cos_phis2 = np.cos(phis2)
    a = haversin(deltas_lats) + cos_phis1 * cos_phis2.T * haversin(deltas_lons)

    vec_dist = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))
    return vec_dist * 1000

In [20]:
def get_list(lat, lon, R, rub):
    point_df = pd.DataFrame({"lat": [lat], "lon": [lon]})
    ls = haversine_2D_mat(point_df, df)[0]
    r = df.loc[ls <= R, "rubrics"]
    rub_split = r.str.split(";").dropna()
    rub_split = rub_split.apply(lambda x: set(x))
    lst = {}
    for sub in rub_split:
        for a in rub:
            fl = False
            for x in a:
                if x in sub:
                    fl = True
            if fl:
                lst["_".join(a)] = lst.get("_".join(a), 0) + 1
    return lst


In [18]:
%%time
R = 5000
rub_lst = get_list(55.810501, 37.518897, R, [["Ресторан", "Кафе"], ["Кафе"], ["Ресторан"]])
rub_lst

CPU times: total: 125 ms
Wall time: 122 ms


{'Ресторан_Кафе': 2691, 'Кафе': 1756, 'Ресторан': 1568}

In [9]:
from collections import Counter
from itertools import chain

st = Counter(chain.from_iterable(df["rubrics"].str.split(";")))
st = Counter({k: v for k, v in st.items() if v >= 30})

In [10]:
st

Counter({'Кафе': 1756,
         'Ресторан': 1568,
         'Салон красоты': 1518,
         'Магазин продуктов': 1032,
         'Ногтевая студия': 1004,
         'Супермаркет': 884,
         'Парикмахерская': 765,
         'Быстрое питание': 716,
         'Бар, паб': 714,
         'Кофейня': 680,
         'Косметология': 575,
         'Медцентр, клиника': 479,
         'Стоматологическая клиника': 450,
         'Салон бровей и ресниц': 410,
         'Магазин одежды': 396,
         'Доставка еды и обедов': 388,
         'Массажный салон': 366,
         'Пункт выдачи': 364,
         'Автосервис, автотехцентр': 363,
         'Гостиница': 338,
         'Аптека': 306,
         'Магазин подарков и сувениров': 302,
         'Банкетный зал': 293,
         'Фитнес-клуб': 286,
         'Магазин алкогольных напитков': 283,
         'Магазин цветов': 279,
         'Магазин хозтоваров и бытовой химии': 273,
         'Кондитерская': 273,
         'Пиццерия': 268,
         'Кофе с собой': 241,
       